<div align="center">

# NB6 · Attack 3: Neutralising Wearable AF Detection

**Explainable Artificial Intelligence in Medical Systems: From Black Box Models to Reliable Clinical Decision Support**<br>
IEEE CARS 2026 Workshop · Grand Forks, ND, USA · 28 October 2026 · Session 11:00 to 11:30

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkukose/xai-cars2026-NB-workshop/blob/main/notebooks/NB6_Attack_Wearable_AF.ipynb)

Prof. Dr. Utku Köse · Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

</div>

## About this notebook

Smartwatches notify their wearers of a possibly irregular rhythm, and the Apple Heart Study enrolled more than 400,000 participants to evaluate such a notification [5]. Atrial fibrillation (AF) often causes no symptoms, yet it multiplies the risk of stroke, and a notification is frequently the first step towards anticoagulation. The detection works on the sequence of inter-beat (RR) intervals: in AF the ventricles respond irregularly, and the tachogram loses its structure. This notebook trains such a detector on real ambulatory ECG annotations, then attacks the RR sequence that feeds it.

**Data.** The MIT-BIH Atrial Fibrillation Database contains 25 long-term ECG recordings of about ten hours from subjects with mostly paroxysmal AF, with expert rhythm annotations [3, 4]. Only the annotation files are downloaded, a few megabytes in total: the rhythm labels and the beat positions from an automated QRS detector. Two records have no beat annotations and are skipped. The data are distributed under the Open Data Commons Attribution License v1.0.

**Threat model.** The attacker alters the RR intervals between the sensor and the classifier, for example through a malicious companion application or a compromised synchronisation service, by a bounded amount per interval. Optical inter-beat intervals are noisier than ECG intervals, particularly during movement, so the question is how much of that noise an attacker needs: the attack starts with 30 ms per interval and escalates through 60, 90, 120, 180 and 240 ms. As in NB4, the attacker observes only the decision: notification or no notification.

The full run takes two to four minutes on a free Colab CPU.

In [ ]:
import importlib.util, subprocess, sys
PACKAGES = {"shap": "shap>=0.46", "gemex": "gemex>=1.2.2", "wfdb": "wfdb>=4.1"}
missing = [spec for module, spec in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import os, io, sys, json, time, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK = os.environ.get("XAI_WORKSHOP_QUICK", "0") == "1"   # smaller runs for automated checks
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
trapezoid = getattr(np, "trapezoid", None) or np.trapz

INK, ALERT, SAFE, MUTED, AMBER, VIOLET = "#1D2742", "#C0392B", "#1F7A6D", "#8A93A6", "#B26B00", "#6C4AB6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
})


def fetch(urls, dest, timeout=120):
    """Download the first reachable URL into dest, reuse an existing copy, and return the path."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in ([urls] if isinstance(urls, str) else urls):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            part = dest.with_name(dest.name + ".part")
            part.write_bytes(payload)
            part.replace(dest)
            return dest
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("No source could be reached:\n" + "\n".join(errors))


print(f"numpy {np.__version__} | pandas {pd.__version__} | quick mode: {QUICK}")

In [ ]:
import shap
import wfdb
import ipywidgets as widgets
from gemex import Explainer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

## 1. Data

In [ ]:
RECORDS = ["04015", "04043", "04048", "04126", "04746", "04908", "04936", "05091", "05121", "05261", "06426",
           "06453", "06995", "07162", "07859", "07879", "07910", "08215", "08219", "08378", "08405", "08434", "08455"]
BASES = ["https://physionet-open.s3.amazonaws.com/afdb/1.0.0/", "https://physionet.org/files/afdb/1.0.0/"]
AFDB = DATA_DIR / "afdb"
for rec in RECORDS:
    for ext in ("hea", "atr", "qrs"):
        fetch([base + f"{rec}.{ext}" for base in BASES], AFDB / f"{rec}.{ext}")


def load_record(rec):
    path = str(AFDB / rec)
    fs = wfdb.rdheader(path).fs
    beats = wfdb.rdann(path, "qrs").sample
    atr = wfdb.rdann(path, "atr")
    notes = np.array([note.strip("\x00 ").strip() for note in atr.aux_note])
    k = np.searchsorted(atr.sample, beats, side="right") - 1
    rhythm = np.where(k >= 0, notes[np.clip(k, 0, None)], "")
    return np.diff(beats) / fs * 1000.0, rhythm[1:]


LEN, STEP = 30, 15
windows, labels, owner = [], [], []
for rec in RECORDS:
    rr, rhythm = load_record(rec)
    if len(rr) < LEN:
        continue
    W = np.lib.stride_tricks.sliding_window_view(rr, LEN)[::STEP]
    R = np.lib.stride_tricks.sliding_window_view(rhythm, LEN)[::STEP]
    af = (R == "(AFIB").mean(axis=1)
    keep = (W.min(axis=1) >= 250) & (W.max(axis=1) <= 2500) & (R != "").all(axis=1) & ((af >= 0.8) | (af <= 0.2))
    windows.append(W[keep])
    labels.append((af[keep] >= 0.8).astype(int))
    owner.append(np.full(keep.sum(), rec))
Xrr, yrr, rec_of = np.vstack(windows), np.concatenate(labels), np.concatenate(owner)
print(f"{len(np.unique(rec_of))} records · {len(Xrr):,} windows of {LEN} beats · {yrr.mean():.0%} in AF")

## 2. An AF notification model

Ten classic heart rate variability features summarise each window of 30 beats: the mean interval, overall and beat-to-beat variability, the share of successive differences above 50 ms, the Poincaré descriptors SD1 and SD2 [6], the turning point ratio, and a robust spread of successive differences. A gradient boosting model learns the rhythm label. The split is by record, so that the test records come from people the model has never seen.

In [ ]:
RR_FEATURES = ["mean_rr", "sdnn", "rmssd", "pnn50", "cv", "sd1", "sd2", "sd1_sd2", "turning_points", "mad_diff"]


def rr_features(W):
    W = np.atleast_2d(np.asarray(W, dtype=float))
    d = np.diff(W, axis=1)
    mean, sdnn = W.mean(axis=1), W.std(axis=1)
    sd1 = np.sqrt(0.5) * d.std(axis=1)
    sd2 = np.sqrt(np.maximum(2 * sdnn ** 2 - sd1 ** 2, 1e-9))
    mid = W[:, 1:-1]
    turns = (((mid > W[:, :-2]) & (mid > W[:, 2:])) | ((mid < W[:, :-2]) & (mid < W[:, 2:]))).mean(axis=1)
    mad = np.median(np.abs(d - np.median(d, axis=1, keepdims=True)), axis=1)
    return np.column_stack([mean, sdnn, np.sqrt((d ** 2).mean(axis=1)), (np.abs(d) > 50).mean(axis=1) * 100,
                            sdnn / mean, sd1, sd2, sd1 / sd2, turns, mad])


recs = np.unique(rec_of)
for attempt in range(50):
    test_recs = rng.choice(recs, size=max(2, int(round(0.3 * len(recs)))), replace=False)
    te = np.isin(rec_of, test_recs)
    if 0 < yrr[te].mean() < 1 and 0 < yrr[~te].mean() < 1:
        break
af_model = make_pipeline(FunctionTransformer(rr_features),
                         GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.1,
                                                    subsample=0.8, random_state=SEED))
af_model.fit(Xrr[~te], yrr[~te])
p_te = af_model.predict_proba(Xrr[te])[:, 1]
X_te, y_te = Xrr[te], yrr[te]
pred = p_te >= 0.5
pd.Series({"test records": ", ".join(sorted(test_recs)), "test AUROC": round(roc_auc_score(y_te, p_te), 3),
           "sensitivity": round(pred[y_te == 1].mean(), 3), "specificity": round(1 - pred[y_te == 0].mean(), 3)}
          ).to_frame("value")

### The attacker's toolkit

The function below implements a decision-based attack in the spirit of HopSkipJump [1]. The attacker never sees the model, its weights or its scores. The only thing observed is the final decision: alert or no alert. The attack proceeds in four moves that repeat:

1. Find any manipulated input without an alert, inside the permitted perturbation box.
2. Walk back towards the genuine input with a binary search until the decision boundary is reached.
3. Estimate the direction of the boundary by probing random nearby points and recording only the yes or no answers.
4. Step along that direction, then search back towards the genuine input again.

Every query is counted, because in a real system each query is an interaction that a defender could, in principle, observe.

In [ ]:
def hsja_attack(decide, x0, lower, upper, init_candidates, n_iter=15, n_dirs=60, seed=0):
    """Decision-based attack in the spirit of HopSkipJump (l2, untargeted, box-constrained).

    decide(X) returns True where the model raises the alert. The search stays inside [lower, upper].
    Returns a dict with the manipulated input (None if no alert-free start exists), distances and query count.
    """
    rs = np.random.default_rng(seed)
    x0, lower, upper = (np.asarray(a, dtype=float) for a in (x0, lower, upper))
    queries = 0

    def alert(X):
        nonlocal queries
        X = np.atleast_2d(X)
        queries += len(X)
        return np.asarray(decide(X), dtype=bool)

    cands = np.clip(np.atleast_2d(np.asarray(init_candidates, dtype=float)), lower, upper)
    free = ~alert(cands)
    if not free.any():
        return dict(x_adv=None, success=False, queries=queries, l2=np.nan, linf=np.nan)
    x_adv = cands[free][np.argmin(np.linalg.norm(cands[free] - x0, axis=1))]

    def to_boundary(x_free, tol=1e-3):
        lo, hi = 0.0, 1.0            # hi always gives an alert-free point on the segment x0 -> x_free
        while hi - lo > tol:
            mid = 0.5 * (lo + hi)
            if alert(x0 + mid * (x_free - x0))[0]:
                lo = mid
            else:
                hi = mid
        return x0 + hi * (x_free - x0)

    xb = to_boundary(x_adv)
    for t in range(1, n_iter + 1):
        dist = np.linalg.norm(xb - x0)
        if dist < 1e-6:
            break
        radius = max(0.1 * dist / np.sqrt(t), 1e-3)
        U = rs.normal(size=(n_dirs, xb.size))
        U /= np.linalg.norm(U, axis=1, keepdims=True)
        phi = np.where(alert(np.clip(xb + radius * U, lower, upper)), -1.0, 1.0)
        v = ((phi - phi.mean())[:, None] * U).mean(axis=0) if phi.std() > 0 else phi[0] * U.mean(axis=0)
        if np.linalg.norm(v) < 1e-12:
            break
        v /= np.linalg.norm(v)
        step = dist / np.sqrt(t)
        for _ in range(12):
            cand = np.clip(xb + step * v, lower, upper)
            if not alert(cand)[0]:
                xb = to_boundary(cand)
                break
            step /= 2
    return dict(x_adv=xb, success=True, queries=queries,
                l2=float(np.linalg.norm(xb - x0)), linf=float(np.abs(xb - x0).max()))

### The defender's toolkit

Three tools turn explanations into security signals. The first is an anomaly score on explanation vectors: a Mahalanobis distance measuring how unusual the SHAP attribution pattern of an input is, compared with genuine inputs that received the same decision. The second is a counterfactual margin: the smallest change in a clinically meaningful direction that would make the alert fire again. It is a counterfactual explanation in the sense of NB2, applied to a "no alert" decision, and it answers the question of how close that decision was to flipping. The third is the Page-Hinkley test [2], a sequential change detector that raises an alarm when the running mean of a monitored signal drifts upwards. Applied to a stream of scores, it looks for a sustained campaign rather than a single suspicious input.

In [ ]:
class MahalanobisDetector:
    """Distance of a vector from a reference population (mean and regularised covariance)."""

    def fit(self, Z):
        Z = np.asarray(Z, dtype=float)
        self.mu = Z.mean(axis=0)
        cov = np.cov(Z, rowvar=False) + 1e-6 * np.eye(Z.shape[1])
        self.precision = np.linalg.pinv(cov)
        return self

    def score(self, Z):
        D = np.asarray(Z, dtype=float) - self.mu
        return np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", D, self.precision, D), 0.0))


def page_hinkley(stream, delta, lam):
    """Page-Hinkley test for an upward shift in the mean. Returns (statistic trace, index of first alarm or None)."""
    mean, cum, cum_min, trace, alarm = 0.0, 0.0, 0.0, [], None
    for t, x in enumerate(stream, start=1):
        mean += (x - mean) / t
        cum += x - mean - delta
        cum_min = min(cum_min, cum)
        trace.append(cum - cum_min)
        if alarm is None and cum - cum_min > lam:
            alarm = t - 1
    return np.array(trace), alarm


def counterfactual_margin(decide, X, move, max_step=1.0, tol=0.01):
    """Smallest step s in [0, max_step] for which decide(move(X, s)) raises the alert; max_step if none does.

    move(X, s) applies a step of size s[i] to row i in a clinically meaningful direction.
    """
    X = np.atleast_2d(np.asarray(X, dtype=float))
    lo, hi = np.zeros(len(X)), np.full(len(X), float(max_step))
    active = np.asarray(decide(move(X, hi)), dtype=bool)
    while np.any(active & (hi - lo > tol)):
        mid = 0.5 * (lo + hi)
        fire = np.asarray(decide(move(X, mid)), dtype=bool)
        hi = np.where(active & fire, mid, hi)
        lo = np.where(active & ~fire, mid, lo)
    return np.where(active, hi, float(max_step))


def detection_auc(scores_clean, scores_attacked):
    """AUROC of a score that should be higher for manipulated inputs."""
    from sklearn.metrics import roc_auc_score
    s = np.concatenate([scores_clean, scores_attacked])
    lab = np.concatenate([np.zeros(len(scores_clean)), np.ones(len(scores_attacked))])
    return roc_auc_score(lab, s)


def campaign(clean_scores, attacked_scores, share=0.2, L=400, delta=0.25, seed=42):
    """Simulate a monitoring stream that switches halfway to a partly manipulated state and run Page-Hinkley.

    Scores are standardised with the median and interquartile range of genuine scores; the alarm threshold is
    set just above the largest statistic seen on 30 streams without any attack.
    """
    rs = np.random.default_rng(seed)
    clean_scores, attacked_scores = np.asarray(clean_scores, float), np.asarray(attacked_scores, float)
    med = np.median(clean_scores)
    iqr = np.subtract(*np.quantile(clean_scores, [0.75, 0.25])) + 1e-9
    z = lambda v: (np.asarray(v) - med) / iqr
    lam = 1.1 * max(page_hinkley(z(rs.choice(clean_scores, L)), delta, np.inf)[0].max() for _ in range(30))
    stream = rs.choice(clean_scores, L)
    t0 = L // 2
    hit = rs.random(L - t0) < share
    stream[t0:][hit] = rs.choice(attacked_scores, hit.sum())
    trace, alarm = page_hinkley(z(stream), delta, lam)
    return z(stream), trace, lam, alarm, t0


def plot_campaigns(signals, share=0.2):
    """signals: list of (title, genuine scores, manipulated scores); higher scores must mean more suspicious."""
    fig, axes = plt.subplots(1, len(signals), figsize=(6.5 * len(signals), 3.4), squeeze=False)
    for ax, (title, sc, sa) in zip(axes[0], signals):
        z_stream, trace_ph, lam, alarm, t0 = campaign(sc, sa, share=share)
        ax.scatter(np.arange(len(z_stream)), z_stream, s=5, alpha=0.5,
                   color=np.where(np.arange(len(z_stream)) >= t0, ALERT, INK))
        ax.axvline(t0, color=ALERT, ls="--", lw=1, label="campaign starts")
        if alarm is not None:
            ax.axvline(alarm, color=AMBER, lw=2, label=f"Page-Hinkley alarm after {alarm - t0} inputs")
        else:
            ax.text(0.02, 0.92, "no alarm", transform=ax.transAxes, color=AMBER, fontweight="bold")
        ax.set_title(title)
        ax.set_xlabel("near-threshold input in the stream")
        ax.set_ylabel("standardised score")
        ax.legend(frameon=False, fontsize=8, loc="lower left")
    plt.tight_layout()
    plt.show()

## 3. The attack

The attack targets AF windows that the model correctly flags. Starting points include a regularised tachogram, in which every interval moves towards the window mean as far as the budget allows, a smoothed tachogram, and random points inside the budget. If no starting point silences the notification, the attack starts again with the next budget; the smallest budget that works is recorded.

In [ ]:
BUDGETS = [30.0, 60.0, 90.0, 120.0, 180.0, 240.0]


def decide(W):
    return af_model.predict_proba(W)[:, 1] >= 0.5


def attack_window(x0, seed):
    total = 0
    for budget in BUDGETS:
        lo, hi = np.maximum(x0 - budget, 300), np.minimum(x0 + budget, 2000)
        regular = x0 + np.clip(x0.mean() - x0, -budget, budget)
        smooth = np.clip(np.convolve(np.pad(x0, 2, mode="edge"), np.ones(5) / 5, mode="valid"), lo, hi)
        init = np.vstack([regular, smooth, rng.uniform(lo, hi, size=(100, LEN))])
        r = hsja_attack(decide, x0, lo, hi, init, n_iter=8 if QUICK else 15, n_dirs=40 if QUICK else 60, seed=seed)
        total += r["queries"]
        if r["success"]:
            return {**r, "queries": total, "budget": budget}
    return {**r, "queries": total, "budget": np.nan}


tp_idx = np.where((y_te == 1) & pred)[0]
attack_idx = rng.choice(tp_idx, size=min(len(tp_idx), 12 if QUICK else 60), replace=False)
t_start = time.time()
att = pd.DataFrame([{**attack_window(X_te[i], seed=k), "index": i, "x0": X_te[i]} for k, i in enumerate(attack_idx)])
ok = att["success"].to_numpy(bool)
assert ok.any(), "No AF window could be silenced within the largest budget."
W_att = np.vstack(att.loc[ok, "x_adv"].to_numpy())
print(f"{len(att)} correct AF notifications attacked in {time.time() - t_start:.0f} s")
table = pd.Series({f"silenced with at most {b:g} ms per interval": f"{(att['budget'] <= b).mean():.0%}" for b in BUDGETS})
table["median largest change per interval (ms), successful attacks"] = round(att.loc[ok, "linf"].median(), 1)
table["median queries per window"] = int(att["queries"].median())
table.to_frame("result")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.step([0] + BUDGETS, [0] + [(att["budget"] <= b).mean() for b in BUDGETS], where="post", color=ALERT, lw=2.2)
ax.set_xlabel("manipulation budget per interval (ms)")
ax.set_ylabel("share of AF notifications silenced")
ax.set_ylim(0, 1.02)
ax.set_title("How much noise does an attacker need?")
plt.tight_layout()
plt.show()

The Poincaré plot places each interval against the next one. Sinus rhythm forms a tight cluster along the diagonal and AF a diffuse cloud; the manipulated window has to look regular enough to the model while staying within the budget of every genuine interval.

In [ ]:
r0 = att[ok].sort_values("budget").iloc[0]
normal = X_te[np.where((y_te == 0) & ~pred)[0][0]]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), gridspec_kw={"width_ratios": [1.7, 1, 1, 1]})
beats = np.arange(LEN)
axes[0].fill_between(beats, r0["x0"] - r0["budget"], r0["x0"] + r0["budget"], color=MUTED, alpha=0.15,
                     label=f"attack budget ({r0['budget']:g} ms)")
axes[0].plot(beats, r0["x0"], "o-", ms=3, color=INK, label=f"genuine AF (P = {af_model.predict_proba(r0['x0'][None])[0, 1]:.2f})")
axes[0].plot(beats, r0["x_adv"], "o-", ms=3, color=ALERT,
             label=f"manipulated (P = {af_model.predict_proba(r0['x_adv'][None])[0, 1]:.2f})")
axes[0].set_xlabel("beat")
axes[0].set_ylabel("RR interval (ms)")
axes[0].legend(frameon=False, fontsize=7)
for ax, (w, title, col) in zip(axes[1:], [(normal, "genuine non-AF", SAFE), (r0["x0"], "genuine AF", INK),
                                          (r0["x_adv"], "manipulated AF", ALERT)]):
    ax.scatter(w[:-1], w[1:], s=14, color=col)
    lims = [min(w.min(), 300), max(w.max(), 1200)]
    ax.plot(lims, lims, color=MUTED, lw=0.8)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_title(title)
    ax.set_xlabel("RR$_n$ (ms)")
axes[1].set_ylabel("RR$_{n+1}$ (ms)")
plt.tight_layout()
plt.show()

## 4. What explanations reveal

The same three signals as in NB4 are computed against genuine non-AF windows whose probability lies just below the decision threshold. For the counterfactual margin, the clinically meaningful direction is more irregularity: every interval moves away from the window mean in proportion to its current deviation, and the margin is the smallest relative increase that brings the notification back.

In [ ]:
booster = af_model[-1]
tree_explainer = shap.TreeExplainer(booster)
band = np.where((y_te == 0) & (p_te >= 0.25) & (p_te < 0.5))[0]
if len(band) < 40:                                    # widen the band if few genuine windows are close to the threshold
    band = np.where((y_te == 0) & (p_te < 0.5))[0][np.argsort(-p_te[(y_te == 0) & (p_te < 0.5)])[:400]]
band = rng.permutation(band)[:3000]
half = len(band) // 2
F_fit, F_test, F_att = rr_features(X_te[band[:half]]), rr_features(X_te[band[half:]]), rr_features(W_att)
S_fit, S_test, S_att = (tree_explainer.shap_values(F) for F in (F_fit, F_test, F_att))
shap_det = MahalanobisDetector().fit(S_fit)
raw_det = MahalanobisDetector().fit(F_fit)
s_clean, s_att = shap_det.score(S_test), shap_det.score(S_att)
print(f"AUROC, SHAP anomaly score: {detection_auc(s_clean, s_att):.3f}")
print(f"AUROC, raw-feature distance: {detection_auc(raw_det.score(F_test), raw_det.score(F_att)):.3f}")


def more_irregular(W, s):
    m = W.mean(axis=1, keepdims=True)
    return np.clip(m + (W - m) * (1 + np.asarray(s)[:, None]), 300, 2000)


m_clean = counterfactual_margin(decide, X_te[band[half:]], more_irregular, max_step=1.0)
m_att = counterfactual_margin(decide, W_att, more_irregular, max_step=1.0)
print(f"median counterfactual margin (relative increase of irregularity): genuine {np.median(m_clean):.2f} · "
      f"manipulated {np.median(m_att):.2f} · AUROC {detection_auc(-m_clean, -m_att):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
diff = pd.Series(S_att.mean(axis=0) - S_test.mean(axis=0), index=RR_FEATURES).sort_values()
axes[0].barh(diff.index, diff.to_numpy(), color=[ALERT if v < 0 else SAFE for v in diff])
axes[0].set_xlabel("mean SHAP, manipulated minus genuine (log-odds)")
axes[0].set_title("Where the attack pushes the explanation")
bins = np.linspace(0, 1, 26)
axes[1].hist(m_clean, bins=bins, color=SAFE, alpha=0.7, density=True, label="genuine, near threshold")
axes[1].hist(m_att, bins=bins, color=ALERT, alpha=0.7, density=True, label="manipulated")
axes[1].set_xlabel("relative increase of irregularity needed to notify")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

**GEMEX view.** GEMEX [8] is applied to the gradient boosting model on the ten variability features, with the training windows as reference.

In [ ]:
gx = Explainer(booster, data_type="tabular", feature_names=RR_FEATURES, class_names=["no AF", "AF"])
reference = rr_features(Xrr[~te][rng.choice((~te).sum(), size=300, replace=False)])
n_g = min(4 if QUICK else 10, len(F_att), len(F_test))
geo_clean = [gx.explain(f, X_reference=reference).geodesic_lengths[-1] for f in F_test[:n_g]]
geo_att = [gx.explain(f, X_reference=reference).geodesic_lengths[-1] for f in F_att[:n_g]]
print(f"median geodesic length: genuine {np.median(geo_clean):.3f} · manipulated {np.median(geo_att):.3f} · "
      f"AUROC {detection_auc(geo_clean, geo_att):.3f} (n = {n_g} each)")
gx.explain(F_att[0], X_reference=reference).plot("gsf_bar", theme="light")
plt.show()

In [ ]:
plot_campaigns([("SHAP anomaly score", s_clean, s_att), ("counterfactual margin (sign flipped)", -m_clean, -m_att)])

**What the evidence supports.** The budget curve shows how much manipulation the detector tolerates before its decision can be bought; the answer is a property of the model that no accuracy figure reports. As in NB4, compare the printed AUROC values: minimal manipulations can leave the attribution pattern ordinary, while the counterfactual margin exposes a notification decision that hangs on a sliver of irregularity. The Poincaré plot is itself an explanation that a cardiologist can read, and it shows what the attack had to do to the rhythm.

## 5. Try it

In [ ]:
def blend(example=0, alpha=1.0):
    r = att[ok].iloc[int(example)]
    w = r["x0"] + alpha * (r["x_adv"] - r["x0"])
    p = af_model.predict_proba(w[None])[0, 1]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [2, 1]})
    axes[0].plot(beats, r["x0"], "o-", ms=3, color=MUTED, label="genuine")
    axes[0].plot(beats, w, "o-", ms=3, color=INK if p >= 0.5 else ALERT, label="reported")
    axes[0].set_title(f"P(AF) = {p:.3f}: {'notification' if p >= 0.5 else 'silent'}")
    axes[0].legend(frameon=False, fontsize=8)
    axes[1].scatter(w[:-1], w[1:], s=12, color=INK if p >= 0.5 else ALERT)
    axes[1].set_title("Poincaré plot")
    plt.tight_layout()
    plt.show()


if ok.any():
    widgets.interact(blend, example=widgets.IntSlider(value=0, min=0, max=int(ok.sum()) - 1, continuous_update=False),
                     alpha=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, continuous_update=False));

## Discussion

1. A consumer notification is not a diagnosis. Does that make the attack less serious, or more, given how many people rely on it?
2. Which party controls the RR series before classification in a typical smartwatch ecosystem, and who is responsible for its integrity?
3. Would an interpretable rule, such as a threshold on RMSSD, be harder or easier to attack than the gradient boosting model?

## Summary

On real ambulatory recordings, a decision-based attacker who shifts inter-beat intervals by bounded amounts can silence correct AF notifications, and the budget curve measures how much noise that takes. The Poincaré view and the counterfactual margin show what the manipulation had to change and how fragile the resulting silence is, and a sequential test lifts these signals to the level of a campaign. Together with NB4 and NB5, the three scenarios show a consistent pattern: attacks that leave validation metrics untouched still leave traces in the explanations. NB7 turns these lessons into governance.

## References

[1] J. Chen, M. I. Jordan, and M. J. Wainwright, "HopSkipJumpAttack: A query-efficient decision-based attack," in *Proc. IEEE Symposium on Security and Privacy (SP)*, 2020, pp. 1277–1294. doi: 10.1109/SP40000.2020.00045

[2] E. S. Page, "Continuous inspection schemes," *Biometrika*, vol. 41, no. 1/2, pp. 100–115, 1954. doi: 10.1093/biomet/41.1-2.100

[3] G. B. Moody and R. G. Mark, "A new method for detecting atrial fibrillation using R-R intervals," *Computers in Cardiology*, vol. 10, pp. 227–230, 1983.

[4] A. L. Goldberger et al., "PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals," *Circulation*, vol. 101, no. 23, pp. e215–e220, 2000. doi: 10.1161/01.CIR.101.23.e215

[5] M. V. Perez et al., "Large-scale assessment of a smartwatch to identify atrial fibrillation," *New England Journal of Medicine*, vol. 381, no. 20, pp. 1909–1917, 2019. doi: 10.1056/NEJMoa1901183

[6] M. Brennan, M. Palaniswami, and P. Kamen, "Do existing measures of Poincaré plot geometry reflect nonlinear features of heart rate variability?," *IEEE Transactions on Biomedical Engineering*, vol. 48, no. 11, pp. 1342–1347, 2001. doi: 10.1109/10.959330

[7] S. M. Lundberg et al., "From local explanations to global understanding with explainable AI for trees," *Nature Machine Intelligence*, vol. 2, pp. 56–67, 2020. doi: 10.1038/s42256-019-0138-9

[8] U. Kose, "GEMEX: Geodesic Entropic Manifold Explainability," Python package, version 1.2.2, 2026. https://pypi.org/project/gemex/

[9] S. G. Finlayson, J. D. Bowers, J. Ito, J. L. Zittrain, A. L. Beam, and I. S. Kohane, "Adversarial attacks on medical machine learning," *Science*, vol. 363, no. 6433, pp. 1287–1289, 2019. doi: 10.1126/science.aaw4399